# Reproduce paper results

Renders the empirical results in the ICD-coding paper from artifacts produced by
the DVC pipeline (`dvc repro evaluate`). The notebook itself does no recomputation;
if an artifact is missing, the corresponding cell fails fast.

## How to run

```bash
dvc repro evaluate              # produces every artifact this notebook reads
uv run jupyter lab coding/eval/reproduce_paper.ipynb
```

Or headless:

```bash
uv run jupyter nbconvert --to notebook --execute \
    coding/eval/reproduce_paper.ipynb \
    --output reproduce_paper.executed.ipynb
```

Outputs land in `outputs/eval/`. The active dataset is set in `params.yaml`.

In [ ]:
# Setup
from __future__ import annotations

import json
from pathlib import Path

import polars as pl

from coding.eval.harness import SPEC
from coding.eval.paths import DATASET_NAME, LABELS_PATH, REPO_ROOT
from coding.eval.paths import EVAL_DIR as OUT_DIR

print(f'REPO_ROOT = {REPO_ROOT}')
print(f'OUT_DIR   = {OUT_DIR}')
print(f'dataset   = {DATASET_NAME} (slug={SPEC.name})')


def load_json(*parts: str) -> dict:
    path = OUT_DIR.joinpath(*parts)
    if not path.exists():
        raise FileNotFoundError(f'Cached artifact missing: {path}')
    return json.loads(path.read_text())


REQUIRED = [
    'metrics_recomputed.parquet',
    'per_cell_cis_recomputed.parquet',
    'bootstrap_recomputed.parquet',
    f'error_analyses/{SPEC.name}/oracle_waterfall.json',
    f'error_analyses/{SPEC.name}/chapter_heatmap_scorable.json',
    f'error_analyses/{SPEC.name}/freq_bands_edin_scorable.json',
    f'error_analyses/{SPEC.name}/delta_attribution.json',
    'stage2_cost_per_note.json',
    'pr_decomp_scorable.csv',
    'comparison_macro_f1.md',
]
missing = [p for p in REQUIRED if not (OUT_DIR / p).exists()]
if missing:
    raise FileNotFoundError(
        f'Missing cached artifacts (run `dvc repro evaluate` first): {missing!r}'
    )
print(f'\nAll {len(REQUIRED)} required cached artifacts present.')

## 1. Table 1 — Datasets and admission counts

Admission-level train/val/test counts pulled from the gold-label parquet (`params.yaml: processed_data_path`).

In [ ]:
def admission_counts(parquet_path: Path):
    if not parquet_path.exists():
        return None
    df = pl.read_parquet(parquet_path)
    id_col = '_id' if '_id' in df.columns else next((c for c in df.columns if c.endswith('_id')), df.columns[0])
    if 'split' in df.columns:
        return df.group_by('split').agg(pl.col(id_col).n_unique().alias('n')).sort('split').to_dicts()
    return [{'split': 'all', 'n': df[id_col].n_unique()}]

print(f'{DATASET_NAME}:', admission_counts(LABELS_PATH))

## 2. Table 2 — Label-space sizes per scope

|C| per InDist / OOD / Full scope, from `per_cell_cis_recomputed.parquet` (`n_in_scope` column under the scorable denominator).

In [ ]:
per_cell = pl.read_parquet(OUT_DIR / 'per_cell_cis_recomputed.parquet')
scope_sizes = (
    per_cell
    .filter(pl.col('macro_denominator') == 'scorable')
    .select(['test_set', 'condition', 'n_in_scope'])
    .unique()
    .sort(['test_set', 'condition'])
)
with pl.Config(tbl_rows=-1):
    print(scope_sizes)

## 3. Table 3 — Main results

Micro-F1 / Macro-F1 with 95% bootstrap CIs across InDist / OOD / Full scopes for five systems.

In [ ]:
tab_main = (
    per_cell
    .filter(pl.col('macro_denominator') == 'scorable')
    .sort(['system', 'condition'])
    .select([
        'system', 'condition',
        'micro_f1', 'micro_ci_lo', 'micro_ci_hi',
        'macro_f1', 'macro_ci_lo', 'macro_ci_hi',
    ])
)
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(tab_main)

In [ ]:
# Headline paired bootstrap deltas
paired = pl.read_parquet(OUT_DIR / 'bootstrap_recomputed.parquet')
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(paired.filter(pl.col('macro_denominator') == 'scorable').sort(['comparison', 'condition']))

## 4. Table 4 — Oracle ablation

Compares real verifier / gap-filler against oracle-grade band verifier and oracle gap-filler. Point estimates only.

In [ ]:
waterfall = load_json('error_analyses', SPEC.name, 'oracle_waterfall.json')
print(f"N={waterfall['n_admissions']:>6,d}  thresholds={waterfall['thresholds']}")
for row in waterfall['rows']:
    full = row['scope_metrics'].get('full', {})
    mi = full.get('f1_micro', float('nan'))
    ma = full.get('f1_macro', float('nan'))
    print(f"  {row['name']:18s}  full·micro={mi:.3f}  full·macro={ma:.3f}")

## 5. Table 5 — Stage-wise decomposition

Cumulative micro/macro per pipeline stage with per-stage Δ and 95% bootstrap CIs.

In [ ]:
stages = (
    per_cell
    .filter(pl.col('macro_denominator') == 'scorable')
    .filter(pl.col('condition') == 'full')
    .filter(pl.col('system').is_in(['sft_only_tau_micro', 'sft_band_verify', 'full_pipeline']))
    .sort('system')
)
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(stages.select(['system', 'micro_f1', 'micro_ci_lo', 'micro_ci_hi', 'macro_f1', 'macro_ci_lo', 'macro_ci_hi']))

# Intermediate (raw gap-fill) — point estimate from the stages_table producer
raw_gap = load_json('results', 'gap_fill_intermediate', SPEC.name, 'raw_gap_fill_scorable.json')
print(f"\n  +gap_fill (raw)  micro={raw_gap['f1_micro']:.3f}  macro={raw_gap['f1_macro']:.3f}")

## 6. Frequency-band F1 ratios (in-text)

From `freq_bands_edin_scorable.json` (`headline` section).

In [ ]:
fb = load_json('error_analyses', SPEC.name, 'freq_bands_edin_scorable.json')
headline = fb.get('headline', {})
print(json.dumps(headline, indent=2, default=str))

---

# Appendix

## A. Stage-2 cost (Table 12)

In [ ]:
cost = load_json('stage2_cost_per_note.json')
print(json.dumps(cost, indent=2, default=str))

## B. OOD per-chapter recovery (Tables 24 / 25)

Per-chapter ΔF1 (`full_vs_sft_high` on the OOD scope) with 95% normal-approx bootstrap CIs.

In [ ]:
ch = load_json('error_analyses', SPEC.name, 'chapter_heatmap_scorable.json')
ood_rows = [
    r for r in ch['chapter_deltas']['rows']
    if r['subset'] == 'ood' and r['transition'] == 'D_total' and r['metric'] == 'f1'
]
ood_rows.sort(key=lambda r: -r['value'])
print(f'{"Chapter":52s}  Δ_F1     [95% CI]              code_type')
print('-' * 100)
for r in ood_rows:
    ci = r.get('ci_normal') or [float('nan'), float('nan')]
    is_proc = r['chapter_id'].startswith('ICD9PCS') or r['chapter_id'].startswith('ICD10PCS')
    code_type = 'PROC' if is_proc else 'DIAG'
    print(f"{r['chapter_name'][:52]:52s}  {r['value']:+.3f}  [{ci[0]:+.3f}, {ci[1]:+.3f}]  {code_type}")

## C. Per-chapter stage heatmaps (Figures 1 / 2)

PNGs written by `coding.eval.render.chapter_stage_figs` to `outputs/eval/figures/`.

In [ ]:
from IPython.display import Image, display

fig_dir = OUT_DIR / 'figures'
for f in (fig_dir / 'fig_chapter_stage_f1_full_diag.png', fig_dir / 'fig_chapter_stage_f1_full_proc.png'):
    print(f'  {f.name}: exists={f.exists()} size={f.stat().st_size if f.exists() else 0} bytes')
    if f.exists():
        display(Image(str(f)))

## D. P/R decomposition (Tables 26 / 27 / 28)

P/R/F1 micro+macro decomposition, ranking metrics (P@k, R@k, R-precision), and calibration.

In [ ]:
pr = pl.read_csv(OUT_DIR / 'pr_decomp_scorable.csv')
with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=80):
    print(pr)

## E. False-positive source by frequency band (Table 22)

In [ ]:
fp_by_band = fb.get('fp_by_band')
if fp_by_band is None:
    print('No `fp_by_band` section. Re-run with `--include-fp-by-band`.')
else:
    print(json.dumps(fp_by_band, indent=2, default=str))

## F. Frequency-stratified F1 (Table 23)

In [ ]:
edin_bins = fb['by_bin']['edin_count']
stages_in_table = ['sft_high', 'sft_post_in_band', 'final']
bands_in_table = ['>1000', '101-1000', '51-100', '11-50', '1-10', '0']
print(f'{"stage":24s} {"band":>10s}  {"micro_F1":>9s}  {"macro_F1":>9s}  {"n_classes":>9s}')
for stage_name in stages_in_table:
    if stage_name not in edin_bins:
        continue
    for band in bands_in_table:
        cell = edin_bins[stage_name].get(band)
        if not isinstance(cell, dict):
            continue
        mi = cell.get('micro_f1', float('nan'))
        ma = cell.get('macro_f1', float('nan'))
        nc = cell.get('n_classes', '-')
        print(f'{stage_name:24s} {band:>10s}  {mi:9.3f}  {ma:9.3f}  {nc:>9}')
    print()

## G. Delta attribution (Table 21)

In [ ]:
delta = load_json('error_analyses', SPEC.name, 'delta_attribution.json')
counts = delta.get('counts', {})
verifier = delta.get('verifier', {})
gap_fill = delta.get('gap_fill', {})
combined = delta.get('combined', {})
print(f"N={delta['n_admissions']:,}, thresholds={delta['thresholds']}")
print(f"verifier  TP={verifier.get('tp', '-'):>6}  FP={verifier.get('fp', '-'):>6}  FP/TP={verifier.get('fp_to_tp_ratio', '-')}")
print(f"gap_fill  TP={gap_fill.get('tp', '-'):>6}  FP={gap_fill.get('fp', '-'):>6}  FP/TP={gap_fill.get('fp_to_tp_ratio', '-')}")
print(f"combined  TP={combined.get('tp', '-'):>6}  FP={combined.get('fp', '-'):>6}  FP/TP={combined.get('fp_to_tp_ratio', '-')}")

## H. Per-class threshold pilot (Table 20)

In [ ]:
pct_path = OUT_DIR / 'results' / 'per_class_threshold_pilot' / SPEC.name / 'per_class_table.csv'
if not pct_path.exists():
    print(f'{pct_path} not found — per-class threshold pilot did not run.')
else:
    pct = pl.read_csv(pct_path)
    with pl.Config(tbl_rows=12, fmt_str_lengths=80):
        print(pct.head(12))
    overall = load_json('results', 'per_class_threshold_pilot', SPEC.name, 'test_metrics.json')
    print(json.dumps(overall, indent=2, default=str))

## I. Macro-denominator side-by-side validation

In [ ]:
print((OUT_DIR / 'comparison_macro_f1.md').read_text())

---

**End of notebook.** Outputs land in `outputs/eval/`. `dvc repro evaluate` regenerates every artifact this notebook reads.